# Laboratory 6: File Handling — Appendix Additional Exercises
**Student Name:** Solis  
**Student ID:** `TUPM-26-2631`  
**Course:** Computer Programming 2 (Strictly for TUP ECE Students)

This notebook contains the complete step-by-step implementations and verified outputs for:
1. Web Server Access Log Analysis
2. Mobile Device Usage Profiling
3. Cybersecurity Incident Report


# Exercise 1: Web Server Access Log Analysis
**Student Name:** Solis  
**Student ID:** `TUPM-26-2631`  

### Mission
Develop a file-based log analyzer that processes a student-specific number of web requests, identifies one HTTP response class, and archives the findings.

### Data Generation Rules
- **Records Processed:** First 1000 + ID_SUM data records (`1000 + 20 = 1020`)
- **Target Status Class:** `(SEED_DIGIT % 4) + 2` = `(1 % 4) + 2 = 3` -> **3xx** (300-399)
- **Log Fields:** `IP`, `Time`, `URL`, `Staus` (dataset column name)


### Step 1: Student Information & Parameter Derivation


In [1]:
LAST_NAME = "Solis"
STUDENT_ID = "TUPM-26-2631"
SEED_DIGIT = int(STUDENT_ID[-1])  # 1
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())  # 20

RECORDS_LIMIT = 1000 + ID_SUM  # 1020
TARGET_CLASS_DIGIT = (SEED_DIGIT % 4) + 2  # 3 -> 3xx
TARGET_STATUS_CLASS = f"{TARGET_CLASS_DIGIT}xx"

print("Student:", LAST_NAME)
print("Student ID:", STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Records to Process (1000 + ID_SUM):", RECORDS_LIMIT)
print("Target Status Class ((SEED_DIGIT % 4) + 2):", TARGET_STATUS_CLASS)


Student: Solis
Student ID: TUPM-26-2631
Seed Digit: 1
ID Sum: 20
Records to Process (1000 + ID_SUM): 1020
Target Status Class ((SEED_DIGIT % 4) + 2): 3xx


### Step 2: Directory & File Path Setup


In [2]:
import os
import csv
import json
import shutil
from collections import Counter
from pathlib import Path

BASE_DIR = Path(".")
DATASET_PATH = BASE_DIR / "datasets" / "weblog.csv"
if not DATASET_PATH.exists():
    DATASET_PATH = BASE_DIR.parent / "datasets" / "weblog.csv"

OUTPUT_DIR = BASE_DIR / "output"
BACKUP_DIR = BASE_DIR / "backup"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

FILTERED_CSV_PATH = OUTPUT_DIR / "weblog_3xx_filtered.csv"
JSON_SUMMARY_PATH = OUTPUT_DIR / "weblog_3xx_summary.json"
BACKUP_JSON_PATH = BACKUP_DIR / "weblog_3xx_summary_backup.json"

print("Dataset Path:", DATASET_PATH.name)
print("Dataset exists:", DATASET_PATH.exists())
print("Output Directory exists:", OUTPUT_DIR.exists())
print("Backup Directory exists:", BACKUP_DIR.exists())


Dataset Path: weblog.csv
Dataset exists: True
Output Directory exists: True
Backup Directory exists: True


### Step 3: Read CSV Line by Line (Without Loading Entire Dataset into Memory)
*Extracts IP address, timestamp, requested URL, and HTTP status code.*


In [3]:
raw_records = []
records_processed = 0
invalid_records = 0

with open(DATASET_PATH, mode="r", encoding="utf-8", errors="ignore") as f:
    reader = csv.DictReader(f)
    print("Dataset Columns:", reader.fieldnames)
    for row in reader:
        if records_processed >= RECORDS_LIMIT:
            break
        records_processed += 1
        
        ip = (row.get("IP") or "").strip()
        time = (row.get("Time") or "").strip()
        url = (row.get("URL") or "").strip()
        status = (row.get("Staus") or row.get("Status") or "").strip()
        
        if not status or not status.isdigit():
            invalid_records += 1
            continue
            
        raw_records.append({
            "IP": ip,
            "Time": time,
            "URL": url,
            "Status": status
        })

print(f"Total records processed: {records_processed}")
print(f"Valid extracted records: {len(raw_records)}")
print(f"Invalid or skipped records: {invalid_records}")


Dataset Columns: ['IP', 'Time', 'URL', 'Staus']
Total records processed: 1020
Valid extracted records: 1020
Invalid or skipped records: 0


### Step 4: Select Records in Target Status Class (3xx)


In [4]:
selected_records = [
    r for r in raw_records 
    if r["Status"].startswith(str(TARGET_CLASS_DIGIT)) and len(r["Status"]) == 3
]

records_in_target = len(selected_records)
print(f"Target status class: {TARGET_STATUS_CLASS}")
print(f"Records matching target class: {records_in_target}")
print("\nSample matching record:")
print(selected_records[0])


Target status class: 3xx
Records matching target class: 164

Sample matching record:
{'IP': '10.128.2.1', 'Time': '[29/Nov/2017:06:59:02', 'URL': 'POST /process.php HTTP/1.1', 'Status': '302'}


### Step 5: Count Requests per Status Code & Top 5 Requested URLs


In [5]:
status_counts = Counter(r["Status"] for r in selected_records)
url_counts = Counter(r["URL"] for r in selected_records)
top_5_urls = url_counts.most_common(5)

print("Requests per Status Code:")
for code, cnt in sorted(status_counts.items()):
    print(f"  Status {code}: {cnt} requests")

print("\nFive Most Frequently Requested URLs:")
for i, (url, cnt) in enumerate(top_5_urls, 1):
    print(f"  {i}. {url} ({cnt} requests)")


Requests per Status Code:
  Status 302: 143 requests
  Status 304: 21 requests

Five Most Frequently Requested URLs:
  1. GET / HTTP/1.1 (45 requests)
  2. POST /process.php HTTP/1.1 (35 requests)
  3. GET /contestproblem.php?name=RUET%20OJ%20Server%20Testing%20Contest HTTP/1.1 (34 requests)
  4. GET /home.php HTTP/1.1 (9 requests)
  5. GET /logout.php HTTP/1.1 (7 requests)


### Step 6: Export Selected Records to CSV and Write Summary to JSON


In [6]:
fieldnames = ["IP", "Time", "URL", "Status"]

with open(FILTERED_CSV_PATH, mode="w", encoding="utf-8", newline="") as cf:
    writer = csv.DictWriter(cf, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(selected_records)

with open(FILTERED_CSV_PATH, mode="r", encoding="utf-8") as cf:
    exported_row_count = sum(1 for _ in csv.DictReader(cf))

summary_data = {
    "assessment_data": {
        "student_name": LAST_NAME,
        "student_id": STUDENT_ID,
        "seed_digit": SEED_DIGIT,
        "id_sum": ID_SUM,
        "records_processed": records_processed,
        "target_status_class": TARGET_STATUS_CLASS,
        "records_in_target_class": records_in_target,
        "invalid_or_skipped_records": invalid_records,
        "count_per_status_code": dict(status_counts),
        "five_most_requested_urls": [
            {"rank": i + 1, "url": u, "count": c}
            for i, (u, c) in enumerate(top_5_urls)
        ],
        "filtered_csv_file": FILTERED_CSV_PATH.name,
        "json_summary_file": JSON_SUMMARY_PATH.name
    }
}

with open(JSON_SUMMARY_PATH, mode="w", encoding="utf-8") as jf:
    json.dump(summary_data, jf, indent=4)

print(f"Exported CSV: {FILTERED_CSV_PATH.name} ({exported_row_count} data rows)")
print(f"Exported JSON: {JSON_SUMMARY_PATH.name} ({JSON_SUMMARY_PATH.stat().st_size} bytes)")


Exported CSV: weblog_3xx_filtered.csv (164 data rows)
Exported JSON: weblog_3xx_summary.json (1316 bytes)


### Step 7: Create Backup of JSON File & Verify It Exists


In [7]:
shutil.copy2(JSON_SUMMARY_PATH, BACKUP_JSON_PATH)

backup_exists = BACKUP_JSON_PATH.exists()
backup_size = BACKUP_JSON_PATH.stat().st_size if backup_exists else 0
backup_verified = backup_exists and backup_size > 0

print("Backup File Path:", BACKUP_JSON_PATH.name)
print("Backup File Exists:", backup_exists)
print(f"Backup File Size: {backup_size} bytes")
print("Backup Verification Status:", "VERIFIED" if backup_verified else "FAILED")


Backup File Path: weblog_3xx_summary_backup.json
Backup File Exists: True
Backup File Size: 1316 bytes
Backup Verification Status: VERIFIED


### Step 8: Assessment Data Table & Final Verification


In [8]:
print("=" * 70)
print(f"{'Item':<35} | {'Result'}")
print("-" * 70)
print(f"{'Records processed':<35} | {records_processed}")
print(f"{'Target status class':<35} | {TARGET_STATUS_CLASS}")
print(f"{'Records in target class':<35} | {records_in_target}")
status_str = ", ".join(f"{k}: {v}" for k, v in sorted(status_counts.items()))
print(f"{'Count per status code':<35} | {status_str}")
print(f"{'Five most requested URLs':<35} |")
for i, (u, c) in enumerate(top_5_urls, 1):
    print(f"  {i}. {u:<50} ({c} requests)")
print(f"{'Filtered CSV file':<35} | {FILTERED_CSV_PATH.name} ({len(selected_records)} rows)")
print(f"{'JSON summary file':<35} | {JSON_SUMMARY_PATH.name}")
print(f"{'Backup verification':<35} | {'VERIFIED' if backup_verified else 'FAILED'} ({BACKUP_JSON_PATH.name}, {backup_size} bytes)")
print("=" * 70)
print("\nFinal Output: Exercise 1 completed successfully!")


Item                                | Result
----------------------------------------------------------------------
Records processed                   | 1020
Target status class                 | 3xx
Records in target class             | 164
Count per status code               | 302: 143, 304: 21
Five most requested URLs            |
  1. GET / HTTP/1.1                                     (45 requests)
  2. POST /process.php HTTP/1.1                         (35 requests)
  3. GET /contestproblem.php?name=RUET%20OJ%20Server%20Testing%20Contest HTTP/1.1 (34 requests)
  4. GET /home.php HTTP/1.1                             (9 requests)
  5. GET /logout.php HTTP/1.1                           (7 requests)
Filtered CSV file                   | weblog_3xx_filtered.csv (164 rows)
JSON summary file                   | weblog_3xx_summary.json
Backup verification                 | VERIFIED (weblog_3xx_summary_backup.json, 1316 bytes)

Final Output: Exercise 1 completed successfully!


---
# 2. Mobile Device Usage Profiling
---


# Exercise 2: Mobile Device Usage Profiling
**Student Name:** Solis  
**Student ID:** `TUPM-26-2631`  

### Mission
Develop a file-based profiling program that filters device records using student-specific criteria and exports a verified usage summary.

### Data Generation Rules
- **Operating System:** Android if SEED_DIGIT is even; iOS if odd -> **iOS** (SEED_DIGIT = 1 is odd)
- **User Behavior Class:** `(SEED_DIGIT % 5) + 1` = `(1 % 5) + 1` = **Class 2**
- **Exported Fields:** `User ID`, `Device Model`, `Operating System`, `Battery Drain (mAh/day)`, `Data Usage (MB/day)`


### Step 1: Student Information & Parameter Derivation


In [9]:
LAST_NAME = "Solis"
STUDENT_ID = "TUPM-26-2631"
SEED_DIGIT = int(STUDENT_ID[-1])  # 1
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())  # 20

TARGET_OS = "Android" if (SEED_DIGIT % 2 == 0) else "iOS"
TARGET_BEHAVIOR_CLASS = (SEED_DIGIT % 5) + 1  # 2

EXPORTED_FIELDS = [
    "User ID",
    "Device Model",
    "Operating System",
    "Battery Drain (mAh/day)",
    "Data Usage (MB/day)",
]

print("Student:", LAST_NAME)
print("Student ID:", STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("Assigned Operating System:", TARGET_OS, "(SEED_DIGIT is odd -> iOS)")
print(f"Assigned User Behavior Class: Class {TARGET_BEHAVIOR_CLASS} ((1 % 5) + 1)")
print("Exported Fields:", EXPORTED_FIELDS)


Student: Solis
Student ID: TUPM-26-2631
Seed Digit: 1
Assigned Operating System: iOS (SEED_DIGIT is odd -> iOS)
Assigned User Behavior Class: Class 2 ((1 % 5) + 1)
Exported Fields: ['User ID', 'Device Model', 'Operating System', 'Battery Drain (mAh/day)', 'Data Usage (MB/day)']


### Step 2: Directory & File Path Setup


In [10]:
import os
import csv
import json
import shutil
from pathlib import Path

BASE_DIR = Path(".")
DATASET_PATH = BASE_DIR / "datasets" / "user_behavior_dataset.csv"
if not DATASET_PATH.exists():
    DATASET_PATH = BASE_DIR.parent / "datasets" / "user_behavior_dataset.csv"

OUTPUT_DIR = BASE_DIR / "output"
BACKUP_DIR = BASE_DIR / "backup"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

EXPORTED_CSV_PATH = OUTPUT_DIR / "user_behavior_ios_class2.csv"
JSON_SUMMARY_PATH = OUTPUT_DIR / "user_behavior_ios_class2_summary.json"
BACKUP_CSV_PATH = BACKUP_DIR / "user_behavior_ios_class2_backup.csv"

print("Dataset Path:", DATASET_PATH.name)
print("Dataset exists:", DATASET_PATH.exists())
print("Output Directory exists:", OUTPUT_DIR.exists())
print("Backup Directory exists:", BACKUP_DIR.exists())


Dataset Path: user_behavior_dataset.csv
Dataset exists: True
Output Directory exists: True
Backup Directory exists: True


### Step 3: Read Dataset & Filter Matching Operating System (iOS)


In [11]:
total_records = 0
os_matched_records = []

with open(DATASET_PATH, mode="r", encoding="utf-8", errors="ignore") as f:
    reader = csv.DictReader(f)
    print("Dataset Columns:", reader.fieldnames)
    for row in reader:
        total_records += 1
        os_val = (row.get("Operating System") or "").strip()
        if os_val.lower() == TARGET_OS.lower():
            os_matched_records.append(row)

records_matching_os = len(os_matched_records)
print(f"Total records in dataset: {total_records}")
print(f"Records matching operating system '{TARGET_OS}': {records_matching_os}")


Dataset Columns: ['User ID', 'Device Model', 'Operating System', 'App Usage Time (min/day)', 'Screen On Time (hours/day)', 'Battery Drain (mAh/day)', 'Number of Apps Installed', 'Data Usage (MB/day)', 'Age', 'Gender', 'User Behavior Class']
Total records in dataset: 700
Records matching operating system 'iOS': 146


### Step 4: Filter Users Belonging to Target Behavior Class (Class 2)


In [12]:
selected_users = []
for row in os_matched_records:
    behavior_class_val = (row.get("User Behavior Class") or "").strip()
    if behavior_class_val == str(TARGET_BEHAVIOR_CLASS):
        selected_users.append(row)

records_in_target_class = len(selected_users)
print(f"Target Behavior Class: Class {TARGET_BEHAVIOR_CLASS}")
print(f"Records in target behavior class: {records_in_target_class}")
print("\nSample selected user record:")
print({f: selected_users[0][f] for f in EXPORTED_FIELDS})


Target Behavior Class: Class 2
Records in target behavior class: 32

Sample selected user record:
{'User ID': '37', 'Device Model': 'iPhone 12', 'Operating System': 'iOS', 'Battery Drain (mAh/day)': '755', 'Data Usage (MB/day)': '573'}


### Step 5: Calculate Average Battery Drain & Mobile Data Usage


In [13]:
battery_drains = []
data_usages = []

for row in selected_users:
    try:
        battery_drains.append(float(row.get("Battery Drain (mAh/day)", 0)))
    except ValueError:
        pass
    try:
        data_usages.append(float(row.get("Data Usage (MB/day)", 0)))
    except ValueError:
        pass

avg_battery = sum(battery_drains) / len(battery_drains) if battery_drains else 0.0
avg_data = sum(data_usages) / len(data_usages) if data_usages else 0.0

print(f"Average Battery Drain: {avg_battery:.2f} mAh/day")
print(f"Average Mobile Data Usage: {avg_data:.2f} MB/day")


Average Battery Drain: 872.03 mAh/day
Average Mobile Data Usage: 451.34 MB/day


### Step 6: Export Selected Users and Specified Fields to CSV


In [14]:
export_data = [
    {field: row.get(field, "").strip() for field in EXPORTED_FIELDS}
    for row in selected_users
]

with open(EXPORTED_CSV_PATH, mode="w", encoding="utf-8", newline="") as cf:
    writer = csv.DictWriter(cf, fieldnames=EXPORTED_FIELDS)
    writer.writeheader()
    writer.writerows(export_data)

with open(EXPORTED_CSV_PATH, mode="r", encoding="utf-8") as cf:
    exported_row_count = sum(1 for _ in csv.DictReader(cf))

print(f"Exported CSV: {EXPORTED_CSV_PATH.name}")
print(f"Exported row count: {exported_row_count}")


Exported CSV: user_behavior_ios_class2.csv
Exported row count: 32


### Step 7: Write JSON Summary & Count Verification


In [15]:
count_verified = (exported_row_count == records_in_target_class)

summary_data = {
    "assessment_data": {
        "student_name": LAST_NAME,
        "student_id": STUDENT_ID,
        "seed_digit": SEED_DIGIT,
        "id_sum": ID_SUM,
        "operating_system_and_behavior_class": f"{TARGET_OS}, Class {TARGET_BEHAVIOR_CLASS}",
        "total_records_in_dataset": total_records,
        "records_matching_operating_system": records_matching_os,
        "records_in_target_behavior_class": records_in_target_class,
        "average_battery_drain_mah_day": round(avg_battery, 2),
        "average_data_usage_mb_day": round(avg_data, 2),
        "count_verification": {
            "selected_record_count": records_in_target_class,
            "exported_row_count": exported_row_count,
            "verified": count_verified
        },
        "exported_csv_file": EXPORTED_CSV_PATH.name,
        "json_summary_file": JSON_SUMMARY_PATH.name
    }
}

with open(JSON_SUMMARY_PATH, mode="w", encoding="utf-8") as jf:
    json.dump(summary_data, jf, indent=4)

print("Count Verification:")
print(f"  Selected records: {records_in_target_class}")
print(f"  Exported rows:    {exported_row_count}")
print(f"  Verification:     {'PASS (VERIFIED)' if count_verified else 'FAIL'}")
print(f"JSON summary saved: {JSON_SUMMARY_PATH.name}")


Count Verification:
  Selected records: 32
  Exported rows:    32
  Verification:     PASS (VERIFIED)
JSON summary saved: user_behavior_ios_class2_summary.json


### Step 8: Create Backup of Exported CSV & Report File Size


In [16]:
shutil.copy2(EXPORTED_CSV_PATH, BACKUP_CSV_PATH)

backup_exists = BACKUP_CSV_PATH.exists()
backup_file_size = BACKUP_CSV_PATH.stat().st_size if backup_exists else 0

summary_data["assessment_data"]["backup_file_and_size"] = {
    "file_name": BACKUP_CSV_PATH.name,
    "size_bytes": backup_file_size,
    "verified": backup_exists
}

with open(JSON_SUMMARY_PATH, mode="w", encoding="utf-8") as jf:
    json.dump(summary_data, jf, indent=4)

print("Backup File:", BACKUP_CSV_PATH.name)
print(f"Backup File Size: {backup_file_size} bytes")
print("Backup Exists:", backup_exists)


Backup File: user_behavior_ios_class2_backup.csv
Backup File Size: 954 bytes
Backup Exists: True


### Step 9: Assessment Data Table & Final Verification


In [17]:
print("=" * 70)
print(f"{'Item':<40} | {'Result'}")
print("-" * 70)
print(f"{'Operating system and behavior class':<40} | {TARGET_OS}, Class {TARGET_BEHAVIOR_CLASS}")
print(f"{'Total records in dataset':<40} | {total_records}")
print(f"{'Records matching operating system':<40} | {records_matching_os}")
print(f"{'Records in target behavior class':<40} | {records_in_target_class}")
print(f"{'Average battery drain (mAh/day)':<40} | {avg_battery:.2f} mAh/day")
print(f"{'Average data usage (MB/day)':<40} | {avg_data:.2f} MB/day")
print(f"{'Count verification':<40} | VERIFIED ({exported_row_count} == {records_in_target_class})")
print(f"{'Exported CSV file':<40} | {EXPORTED_CSV_PATH.name}")
print(f"{'JSON summary file':<40} | {JSON_SUMMARY_PATH.name}")
print(f"{'Backup file and size':<40} | {BACKUP_CSV_PATH.name} ({backup_file_size} bytes)")
print("=" * 70)
print("\nFinal Output: Exercise 2 completed successfully!")


Item                                     | Result
----------------------------------------------------------------------
Operating system and behavior class      | iOS, Class 2
Total records in dataset                 | 700
Records matching operating system        | 146
Records in target behavior class         | 32
Average battery drain (mAh/day)          | 872.03 mAh/day
Average data usage (MB/day)              | 451.34 MB/day
Count verification                       | VERIFIED (32 == 32)
Exported CSV file                        | user_behavior_ios_class2.csv
JSON summary file                        | user_behavior_ios_class2_summary.json
Backup file and size                     | user_behavior_ios_class2_backup.csv (954 bytes)

Final Output: Exercise 2 completed successfully!


---
# 3. Cybersecurity Incident Report
---


# Exercise 3: Cybersecurity Incident Report
**Student Name:** Solis  
**Student ID:** `TUPM-26-2631`  

### Mission
Develop a file-based incident reporting program that processes records for a student-specific year, summarizes attack types, and archives the findings.

### Data Generation Rules
- **Target Year:** `2015 + SEED_DIGIT` = `2015 + 1 = 2016`
- **Loss Field:** `Financial Loss (in Million $)`
- **Count Field:** `Attack Type`


### Step 1: Student Information & Parameter Derivation


In [18]:
LAST_NAME = "Solis"
STUDENT_ID = "TUPM-26-2631"
SEED_DIGIT = int(STUDENT_ID[-1])  # 1
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())  # 20

TARGET_YEAR = 2015 + SEED_DIGIT  # 2015 + 1 = 2016
LOSS_FIELD = "Financial Loss (in Million $)"
COUNT_FIELD = "Attack Type"

print("Student:", LAST_NAME)
print("Student ID:", STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print(f"Target Year (2015 + SEED_DIGIT): {TARGET_YEAR}")
print("Loss Field:", LOSS_FIELD)
print("Count Field:", COUNT_FIELD)


Student: Solis
Student ID: TUPM-26-2631
Seed Digit: 1
Target Year (2015 + SEED_DIGIT): 2016
Loss Field: Financial Loss (in Million $)
Count Field: Attack Type


### Step 2: Directory & File Path Setup


In [19]:
import os
import csv
import json
import shutil
from collections import Counter
from pathlib import Path

BASE_DIR = Path(".")
DATASET_PATH = BASE_DIR / "datasets" / "Global_Cybersecurity_Threats_2015-2024.csv"
if not DATASET_PATH.exists():
    DATASET_PATH = BASE_DIR.parent / "datasets" / "Global_Cybersecurity_Threats_2015-2024.csv"

OUTPUT_DIR = BASE_DIR / "output"
BACKUP_DIR = BASE_DIR / "backup"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

FILTERED_CSV_PATH = OUTPUT_DIR / "cybersecurity_2016_filtered.csv"
JSON_TOP5_PATH = OUTPUT_DIR / "cybersecurity_2016_top5_costliest.json"
BACKUP_JSON_PATH = BACKUP_DIR / "cybersecurity_2016_top5_costliest_backup.json"
REPORT_TXT_PATH = OUTPUT_DIR / "cybersecurity_2016_incident_report.txt"

print("Dataset Path:", DATASET_PATH.name)
print("Dataset exists:", DATASET_PATH.exists())
print("Output Directory exists:", OUTPUT_DIR.exists())
print("Backup Directory exists:", BACKUP_DIR.exists())


Dataset Path: Global_Cybersecurity_Threats_2015-2024.csv
Dataset exists: True
Output Directory exists: True
Backup Directory exists: True


### Step 3: Read Dataset & Select Incidents in Target Year (2016)


In [20]:
total_records = 0
selected_incidents = []
dataset_fieldnames = []

with open(DATASET_PATH, mode="r", encoding="utf-8", errors="ignore") as f:
    reader = csv.DictReader(f)
    dataset_fieldnames = reader.fieldnames
    print("Dataset Columns:", dataset_fieldnames)
    
    order = 0
    for row in reader:
        total_records += 1
        year_val = (row.get("Year") or "").strip()
        if year_val == str(TARGET_YEAR):
            order += 1
            loss_str = (row.get(LOSS_FIELD) or "").strip()
            try:
                loss_val = float(loss_str)
            except ValueError:
                loss_val = 0.0
            
            entry = dict(row)
            entry["_orig_order"] = order
            entry["_loss_float"] = loss_val
            selected_incidents.append(entry)

incidents_target_year = len(selected_incidents)
print(f"Total records in dataset: {total_records}")
print(f"Incidents occurring in target year {TARGET_YEAR}: {incidents_target_year}")


Dataset Columns: ['Country', 'Year', 'Attack Type', 'Target Industry', 'Financial Loss (in Million $)', 'Number of Affected Users', 'Attack Source', 'Security Vulnerability Type', 'Defense Mechanism Used', 'Incident Resolution Time (in Hours)']
Total records in dataset: 3000
Incidents occurring in target year 2016: 285


### Step 4: Count Incidents Belonging to Each Attack Type


In [21]:
attack_counts = Counter(inc.get(COUNT_FIELD, "Unknown").strip() for inc in selected_incidents)

print(f"Attack Types and Incident Counts for {TARGET_YEAR}:")
for attack_type, count in attack_counts.most_common():
    percentage = (count / incidents_target_year) * 100
    print(f"  - {attack_type:<20}: {count:>3} incidents ({percentage:>5.1f}%)")


Attack Types and Incident Counts for 2016:
  - Phishing            :  55 incidents ( 19.3%)
  - DDoS                :  53 incidents ( 18.6%)
  - Man-in-the-Middle   :  47 incidents ( 16.5%)
  - SQL Injection       :  47 incidents ( 16.5%)
  - Ransomware          :  42 incidents ( 14.7%)
  - Malware             :  41 incidents ( 14.4%)


### Step 5: Calculate Total & Average Financial Loss


In [22]:
losses = [inc["_loss_float"] for inc in selected_incidents]
total_financial_loss = sum(losses)
avg_financial_loss = (total_financial_loss / len(losses)) if losses else 0.0

print(f"Total Financial Loss ({TARGET_YEAR}): ${total_financial_loss:,.2f} Million USD")
print(f"Average Financial Loss per Incident: ${avg_financial_loss:,.2f} Million USD")


Total Financial Loss (2016): $13,947.26 Million USD
Average Financial Loss per Incident: $48.94 Million USD


### Step 6: Rank Incidents by Financial Loss & Identify Top 5 Costliest


In [23]:
ranked_incidents = sorted(
    selected_incidents,
    key=lambda x: (-x["_loss_float"], x["_orig_order"])
)
top_5_costliest = ranked_incidents[:5]

clean_top_5 = []
print("Top 5 Costliest Incidents:")
for rank, inc in enumerate(top_5_costliest, 1):
    item = {k: v for k, v in inc.items() if not k.startswith("_")}
    item["Rank"] = rank
    clean_top_5.append(item)
    print(f"  {rank}. {item['Attack Type']} | Loss: ${item[LOSS_FIELD]}M | Country: {item['Country']} | Target: {item['Target Industry']}")


Top 5 Costliest Incidents:
  1. Phishing | Loss: $99.98M | Country: Germany | Target: Telecommunications
  2. Ransomware | Loss: $99.88M | Country: USA | Target: Telecommunications
  3. Malware | Loss: $99.29M | Country: Brazil | Target: IT
  4. Ransomware | Loss: $99.19M | Country: USA | Target: Healthcare
  5. Phishing | Loss: $98.72M | Country: Australia | Target: Education


### Step 7: Export Top 5 to JSON & Export Filtered Incidents to CSV


In [24]:
json_export_data = {
    "assessment_data": {
        "student_name": LAST_NAME,
        "student_id": STUDENT_ID,
        "seed_digit": SEED_DIGIT,
        "id_sum": ID_SUM,
        "target_year": TARGET_YEAR,
        "total_records_in_dataset": total_records,
        "incidents_in_target_year": incidents_target_year,
        "count_per_attack_type": dict(attack_counts),
        "total_financial_loss_usd_millions": round(total_financial_loss, 2),
        "average_financial_loss_usd_millions": round(avg_financial_loss, 2),
        "five_costliest_incidents": clean_top_5
    }
}

with open(JSON_TOP5_PATH, mode="w", encoding="utf-8") as jf:
    json.dump(json_export_data, jf, indent=4)

csv_fieldnames = [k for k in dataset_fieldnames if not k.startswith("_")]
with open(FILTERED_CSV_PATH, mode="w", encoding="utf-8", newline="") as cf:
    writer = csv.DictWriter(cf, fieldnames=csv_fieldnames)
    writer.writeheader()
    for inc in selected_incidents:
        writer.writerow({k: inc[k] for k in csv_fieldnames})

print(f"Exported JSON Top 5: {JSON_TOP5_PATH.name} ({JSON_TOP5_PATH.stat().st_size} bytes)")
print(f"Exported Filtered CSV: {FILTERED_CSV_PATH.name} ({len(selected_incidents)} records)")


Exported JSON Top 5: cybersecurity_2016_top5_costliest.json (3523 bytes)
Exported Filtered CSV: cybersecurity_2016_filtered.csv (285 records)


### Step 8: Create Verified Backup of JSON File


In [25]:
shutil.copy2(JSON_TOP5_PATH, BACKUP_JSON_PATH)

backup_exists = BACKUP_JSON_PATH.exists()
backup_size = BACKUP_JSON_PATH.stat().st_size if backup_exists else 0
backup_verified = backup_exists and backup_size > 0

print("Backup File Path:", BACKUP_JSON_PATH.name)
print("Backup File Exists:", backup_exists)
print(f"Backup File Size: {backup_size} bytes")
print("Backup Verification Status:", "VERIFIED" if backup_verified else "FAILED")


Backup File Path: cybersecurity_2016_top5_costliest_backup.json
Backup File Exists: True
Backup File Size: 3523 bytes
Backup Verification Status: VERIFIED


### Step 9: Generate Comprehensive Text Report


In [26]:
report_lines = [
    "=" * 70,
    f"CYBERSECURITY INCIDENT REPORT - FISCAL YEAR {TARGET_YEAR}",
    "=" * 70,
    f"Student: {LAST_NAME} ({STUDENT_ID})",
    f"Seed Digit: {SEED_DIGIT} | ID Sum: {ID_SUM}",
    f"Target Year: {TARGET_YEAR}",
    "-" * 70,
    "1. EXECUTIVE SUMMARY & ASSESSMENT METRICS",
    f"  Total records in dataset:             {total_records}",
    f"  Incidents occurring in {TARGET_YEAR}:            {incidents_target_year}",
    f"  Total Financial Loss:                 ${total_financial_loss:,.2f} Million USD",
    f"  Average Financial Loss per Incident:  ${avg_financial_loss:,.2f} Million USD",
    "-" * 70,
    "2. INCIDENTS BY ATTACK TYPE",
]
for att, count in sorted(attack_counts.items(), key=lambda x: -x[1]):
    pct = (count / incidents_target_year * 100) if incidents_target_year else 0
    report_lines.append(f"  - {att:<22}: {count:>3} incidents ({pct:>5.1f}%)")

report_lines.extend([
    "-" * 70,
    "3. FIVE COSTLIEST INCIDENTS OF 2016",
])
for item in clean_top_5:
    report_lines.append(
        f"  Rank {item['Rank']}: Loss = ${item[LOSS_FIELD]} Million USD\n"
        f"         Attack Type:     {item['Attack Type']}\n"
        f"         Country:         {item['Country']}\n"
        f"         Target Industry: {item['Target Industry']}\n"
        f"         Affected Users:  {item['Number of Affected Users']}\n"
        f"         Defense Used:    {item['Defense Mechanism Used']}\n"
        f"         Vulnerability:   {item['Security Vulnerability Type']}\n"
    )

report_lines.extend([
    "-" * 70,
    "4. OUTPUT & ARCHIVE VERIFICATION",
    f"  Filtered CSV File:    {FILTERED_CSV_PATH.name} ({len(selected_incidents)} records)",
    f"  JSON Summary File:    {JSON_TOP5_PATH.name} ({JSON_TOP5_PATH.stat().st_size} bytes)",
    f"  Backup JSON File:     {BACKUP_JSON_PATH.name} ({backup_size} bytes)",
    f"  Backup Verification:  {'VERIFIED' if backup_verified else 'FAILED'}",
    f"  Incident Report File: {REPORT_TXT_PATH.name}",
    "=" * 70,
])

report_text = "\n".join(report_lines)
with open(REPORT_TXT_PATH, mode="w", encoding="utf-8") as rf:
    rf.write(report_text)

print(f"Generated Text Report: {REPORT_TXT_PATH.name}")
print(f"Report Size: {REPORT_TXT_PATH.stat().st_size} bytes")


Generated Text Report: cybersecurity_2016_incident_report.txt
Report Size: 2956 bytes


### Step 10: Assessment Data Table & Final Verification


In [27]:
print("=" * 70)
print(f"{'Item':<38} | {'Result'}")
print("-" * 70)
print(f"{'Target year':<38} | {TARGET_YEAR}")
print(f"{'Total records in dataset':<38} | {total_records}")
print(f"{'Incidents in target year':<38} | {incidents_target_year}")
attack_summary_str = ", ".join(f"{k}: {v}" for k, v in sorted(attack_counts.items(), key=lambda x: -x[1]))
print(f"{'Count per attack type':<38} | {attack_summary_str}")
print(f"{'Total financial loss (USD millions)':<38} | ${total_financial_loss:,.2f} Million")
print(f"{'Average financial loss (USD millions)':<38} | ${avg_financial_loss:,.2f} Million")
print(f"{'Five costliest incidents':<38} |")
for item in clean_top_5:
    print(f"  {item['Rank']}. {item['Attack Type']} - ${item[LOSS_FIELD]}M - {item['Country']} ({item['Target Industry']})")
print(f"{'Filtered CSV file':<38} | {FILTERED_CSV_PATH.name} ({len(selected_incidents)} records)")
print(f"{'JSON summary file':<38} | {JSON_TOP5_PATH.name}")
print(f"{'Backup verification':<38} | {'VERIFIED' if backup_verified else 'FAILED'} ({BACKUP_JSON_PATH.name}, {backup_size} bytes)")
print(f"{'Report file':<38} | {REPORT_TXT_PATH.name}")
print("=" * 70)
print("\nFinal Output: Exercise 3 completed successfully!")


Item                                   | Result
----------------------------------------------------------------------
Target year                            | 2016
Total records in dataset               | 3000
Incidents in target year               | 285
Count per attack type                  | Phishing: 55, DDoS: 53, Man-in-the-Middle: 47, SQL Injection: 47, Ransomware: 42, Malware: 41
Total financial loss (USD millions)    | $13,947.26 Million
Average financial loss (USD millions)  | $48.94 Million
Five costliest incidents               |
  1. Phishing - $99.98M - Germany (Telecommunications)
  2. Ransomware - $99.88M - USA (Telecommunications)
  3. Malware - $99.29M - Brazil (IT)
  4. Ransomware - $99.19M - USA (Healthcare)
  5. Phishing - $98.72M - Australia (Education)
Filtered CSV file                      | cybersecurity_2016_filtered.csv (285 records)
JSON summary file                      | cybersecurity_2016_top5_costliest.json
Backup verification                    | VERIFI